In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["ANTHROPIC_API_KEY"] = os.getenv("ANTHROPIC_API_KEY")

# Custom middleware
Build custom middleware by implementing hooks that run at specific points in the agent execution flow.

## Hooks
Middleware provides two styles of hooks to intercept agent execution:
- Node-style hooks
- Wrap-style hooks

### 1: Node-style hooks
Run sequentially at specific execution points. Use for logging, validation, and state updates.
""Available hooks:""
- `before_agent` - Before agent starts (once per invocation)
- `before_model` - Before each model call
- `after_model` - After each model response
- `after_agent` - After agent finishes (once per invocation)

### Examples : By using Decoraters

```python
from langchain.agents.middleware import before_model, after_model, AgentState
from langchain.messages import AIMessage
from langgraph.runtime import Runtime
from typing import Any

In [4]:
from langchain.agents import create_agent
from langchain.agents.middleware import before_model, after_model, AgentState
from langchain.messages import AIMessage, HumanMessage
from langgraph.runtime import Runtime
from langchain_anthropic import ChatAnthropic
from typing import Any


# --------------------------------
# Cheap Claude Model
# --------------------------------
model = ChatAnthropic(
    model="claude-3-haiku-20240307",
    temperature=0,
    max_tokens=20,
)


# --------------------------------
# Middleware 1: Conversation Limit
# --------------------------------
@before_model(can_jump_to=["end"])
def check_message_limit(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:

    print("Current message count:", len(state["messages"]))

    if len(state["messages"]) >= 6:

        return {
            "messages": [
                AIMessage(
                    content="Conversation limit reached. Please start a new chat."
                )
            ],
            "jump_to": "end",
        }

    return None


# --------------------------------
# Middleware 2: Log Model Response
# --------------------------------
@after_model
def log_response(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:

    print("\n[LOG] Model Response:")
    print(state["messages"][-1].content)

    return None


# --------------------------------
# Create Agent
# --------------------------------
agent = create_agent(
    model=model,
    tools=[],
    middleware=[check_message_limit, log_response],
)


# --------------------------------
# Conversation Memory
# --------------------------------
chat_history = []


def run_agent(prompt):

    print("\nUSER:", prompt)

    chat_history.append(HumanMessage(content=prompt))

    result = agent.invoke({"messages": chat_history})

    response = result["messages"][-1]

    chat_history.append(response)

    print("AI:", response.content)


# --------------------------------
# Real Life Test Cases
# --------------------------------
tests = [
    "What is your refund policy?",
    "How long does delivery take?",
    "Do you offer international shipping?",
    "What payment methods are accepted?",
    "Can I track my order?",
    "How do I cancel an order?",
]


for i, t in enumerate(tests):

    print("\n----------------------")
    print("TEST", i + 1)

    run_agent(t)


----------------------
TEST 1

USER: What is your refund policy?
Current message count: 1

[LOG] Model Response:
I don't actually have a refund policy. I'm Claude, an AI assistant created by
AI: I don't actually have a refund policy. I'm Claude, an AI assistant created by

----------------------
TEST 2

USER: How long does delivery take?
Current message count: 3

[LOG] Model Response:
I don't actually handle any deliveries or have information about shipping times. I'm an AI assistant
AI: I don't actually handle any deliveries or have information about shipping times. I'm an AI assistant

----------------------
TEST 3

USER: Do you offer international shipping?
Current message count: 5

[LOG] Model Response:
I'm afraid I don't have any information about shipping or delivery, as I'm an AI assistant
AI: I'm afraid I don't have any information about shipping or delivery, as I'm an AI assistant

----------------------
TEST 4

USER: What payment methods are accepted?
Current message count: 

### Examples : By using Classes

```python
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langchain.messages import AIMessage
from langgraph.runtime import Runtime
from typing import Any

In [5]:
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langchain.messages import AIMessage, HumanMessage
from langgraph.runtime import Runtime
from langchain_anthropic import ChatAnthropic
from typing import Any


# --------------------------------
# Cheap Claude Model
# --------------------------------
model = ChatAnthropic(
    model="claude-3-haiku-20240307",
    temperature=0,
    max_tokens=10,
)


# --------------------------------
# Custom Middleware (Class Style)
# --------------------------------
class MessageLimitMiddleware(AgentMiddleware):

    def __init__(self, max_messages: int = 6):
        super().__init__()
        self.max_messages = max_messages

    # Runs before model call
    @hook_config(can_jump_to=["end"])
    def before_model(
        self, state: AgentState, runtime: Runtime
    ) -> dict[str, Any] | None:

        print("Current message count:", len(state["messages"]))

        if len(state["messages"]) >= self.max_messages:

            return {
                "messages": [
                    AIMessage(
                        content="Conversation limit reached. Please start a new chat."
                    )
                ],
                "jump_to": "end",
            }

        return None

    # Runs after model response
    def after_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:

        print("\n[LOG] Model Response:")
        print(state["messages"][-1].content)

        return None


# --------------------------------
# Create Agent
# --------------------------------
agent = create_agent(
    model=model,
    tools=[],
    middleware=[MessageLimitMiddleware(max_messages=6)],
)


# --------------------------------
# Conversation Memory
# --------------------------------
chat_history = []


def run_agent(prompt):

    print("\nUSER:", prompt)

    chat_history.append(HumanMessage(content=prompt))

    result = agent.invoke({"messages": chat_history})

    response = result["messages"][-1]

    chat_history.append(response)

    print("AI:", response.content)


# --------------------------------
# Real Life Test Cases
# --------------------------------
tests = [
    "What is your refund policy?",
    "How long does delivery take?",
    "Do you offer international shipping?",
    "What payment methods are accepted?",
    "Can I track my order?",
    "How do I cancel an order?",
]


for i, t in enumerate(tests):

    print("\n----------------------")
    print("TEST", i + 1)

    run_agent(t)


----------------------
TEST 1

USER: What is your refund policy?
Current message count: 1

[LOG] Model Response:
I don't actually have a refund policy.
AI: I don't actually have a refund policy.

----------------------
TEST 2

USER: How long does delivery take?
Current message count: 3

[LOG] Model Response:
I don't actually handle any deliveries or have
AI: I don't actually handle any deliveries or have

----------------------
TEST 3

USER: Do you offer international shipping?
Current message count: 5

[LOG] Model Response:
I don't actually offer any shipping or have the
AI: I don't actually offer any shipping or have the

----------------------
TEST 4

USER: What payment methods are accepted?
Current message count: 7
AI: Conversation limit reached. Please start a new chat.

----------------------
TEST 5

USER: Can I track my order?
Current message count: 9
AI: Conversation limit reached. Please start a new chat.

----------------------
TEST 6

USER: How do I cancel an order?
Current

### 2: Wrap-style hooks
Intercept execution and control when the handler is called. Use for retries, caching, and transformation.
You decide if the handler is called zero times (short-circuit), once (normal flow), or multiple times (retry logic).
""Available hooks:""
- `wrap_model_call` - Around each model call
- `wrap_tool_call` - Arounf each tool call


### Examples : By using Decoraters

```python
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

In [7]:
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.messages import HumanMessage
from langchain_anthropic import ChatAnthropic
from typing import Callable


# --------------------------------
# Cheap Claude Model
# --------------------------------
model = ChatAnthropic(
    model="claude-3-haiku-20240307",
    temperature=0,
    max_tokens=20,
)


# --------------------------------
# Retry Middleware
# --------------------------------
@wrap_model_call
def retry_model(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:

    max_retries = 3

    # Access the user message
    user_message = request.messages[-1].content

    for attempt in range(max_retries):

        try:

            print(f"\nAttempt {attempt + 1} calling model...")

            # --------------------------------
            # Forced failure condition
            # --------------------------------
            if "force failure" in user_message.lower():
                raise RuntimeError("Simulated model failure")

            response = handler(request)

            print("Model call successful")

            return response

        except Exception as e:

            if attempt == max_retries - 1:

                print("Final failure after retries")

                raise

            print(f"Retry {attempt + 1}/{max_retries} after error: {e}")


# --------------------------------
# Create Agent
# --------------------------------
agent = create_agent(
    model=model,
    tools=[],
    middleware=[retry_model],
)


# --------------------------------
# Conversation Memory
# --------------------------------
chat_history = []


def run_agent(prompt):

    print("\nUSER:", prompt)

    chat_history.append(HumanMessage(content=prompt))

    try:

        result = agent.invoke({"messages": chat_history})

        response = result["messages"][-1]

        chat_history.append(response)

        print("AI:", response.content)

    except Exception as e:

        print("FINAL ERROR:", e)


# --------------------------------
# Test Cases
# --------------------------------
tests = [
    "Explain machine learning simply",
    "What is LangChain used for?",
    "force failure",  # <-- failing test case
]


for i, t in enumerate(tests):

    print("\n----------------------")
    print("TEST", i + 1)

    run_agent(t)


----------------------
TEST 1

USER: Explain machine learning simply

Attempt 1 calling model...
Model call successful
AI: Machine learning is a field of artificial intelligence that allows computers to learn and improve from experience without being explicitly

----------------------
TEST 2

USER: What is LangChain used for?

Attempt 1 calling model...
Model call successful
AI: LangChain is a framework for building applications with large language models (LLMs) like

----------------------
TEST 3

USER: force failure

Attempt 1 calling model...
Retry 1/3 after error: Simulated model failure

Attempt 2 calling model...
Retry 2/3 after error: Simulated model failure

Attempt 3 calling model...
Final failure after retries
FINAL ERROR: Simulated model failure


### Examples : By using Classes

```python
from langchain.agents.middleware import AgentMiddleware, ModelRequest, ModelResponse
from typing import Callable

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware, ModelRequest, ModelResponse
from langchain.messages import HumanMessage
from langchain_anthropic import ChatAnthropic
from typing import Callable


# --------------------------------
# Cheap Claude Model
# --------------------------------
model = ChatAnthropic(
    model="claude-3-haiku-20240307",
    temperature=0,
    max_tokens=20,
)


# --------------------------------
# Retry Middleware (Class Style)
# --------------------------------
class RetryMiddleware(AgentMiddleware):

    def __init__(self, max_retries: int = 3):
        super().__init__()
        self.max_retries = max_retries

    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse:

        user_message = request.messages[-1].content

        for attempt in range(self.max_retries):

            try:

                print(f"\nAttempt {attempt + 1} calling model...")

                # --------------------------------
                # Forced failure condition
                # --------------------------------
                if "force failure" in user_message.lower():
                    raise RuntimeError("Simulated model crash")

                response = handler(request)

                print("Model call successful")

                return response

            except Exception as e:

                if attempt == self.max_retries - 1:

                    print("Final failure after retries")

                    raise

                print(f"Retry {attempt + 1}/{self.max_retries} after error: {e}")


# --------------------------------
# Create Agent
# --------------------------------
agent = create_agent(
    model=model,
    tools=[],
    middleware=[RetryMiddleware(max_retries=3)],
)


# --------------------------------
# Conversation Memory
# --------------------------------
chat_history = []


def run_agent(prompt):

    print("\nUSER:", prompt)

    chat_history.append(HumanMessage(content=prompt))

    try:

        result = agent.invoke({"messages": chat_history})

        response = result["messages"][-1]

        chat_history.append(response)

        print("AI:", response.content)

    except Exception as e:

        print("FINAL ERROR:", e)


# --------------------------------
# Test Cases
# --------------------------------
tests = [
    "Explain machine learning simply",
    "What is LangChain used for?",
    "force failure",  # failing test case
]


for i, t in enumerate(tests):

    print("\n----------------------")
    print("TEST", i + 1)

    run_agent(t)


----------------------
TEST 1

USER: Explain machine learning simply

Attempt 1 calling model...
Model call successful
AI: Machine learning is a field of artificial intelligence that allows computers to learn and improve from experience without being explicitly

----------------------
TEST 2

USER: What is LangChain used for?

Attempt 1 calling model...
Model call successful
AI: LangChain is a framework for building applications with large language models (LLMs) like

----------------------
TEST 3

USER: force failure

Attempt 1 calling model...
Retry 1/3 after error: Simulated model crash

Attempt 2 calling model...
Retry 2/3 after error: Simulated model crash

Attempt 3 calling model...
Final failure after retries
FINAL ERROR: Simulated model crash


# LangChain Middleware — State Updates

In LangChain agents, **middleware hooks can modify the agent state during execution**.

Agent state is a shared data structure that can store information like:

- messages
- tool outputs
- metadata
- token usage
- custom fields

Middleware can update this state in **two different ways** depending on the hook type.

---

# Two Types of Hooks That Update State

| Hook Type | Update Method |
|---|---|
| Node-style hooks | Return a `dict` |
| Wrap-style hooks | Return `Command` inside response |

---



**Node Hooks**

```python
return {"key": value}
```

**Wrap Hooks**

```python
return response.with_command(
    Command(update={"key": value})
)
```

Node hooks are **simpler**, while wrap hooks provide **full control over execution**.

# 1️⃣ Node-Style Hooks

Node-style hooks run at **specific points in the agent execution pipeline**.

Examples:

- `before_agent`
- `before_model`
- `after_model`
- `after_agent`

To update the agent state, simply **return a dictionary**.

LangGraph automatically merges the returned dictionary into the **agent state using reducers**.

---

# Node Hook Execution Flow

```
Agent Start
   ↓
before_agent
   ↓
before_model
   ↓
Model Call
   ↓
after_model
   ↓
after_agent
```

---

# Example: Update State in a Node Hook

Example: **Count how many times the model is called**

```python
from langchain.agents.middleware import before_model, AgentState
from langgraph.runtime import Runtime
from typing import Any


@before_model
def count_model_calls(
    state: AgentState,
    runtime: Runtime
) -> dict[str, Any] | None:

    calls = state.get("model_calls", 0)

    return {
        "model_calls": calls + 1
    }
```

### What Happens

1. Middleware reads the `model_calls` value from state  
2. Increments the count  
3. Returns a dictionary  
4. LangGraph merges it into the agent state  

---

# Example: Track Last AI Response

```python
from langchain.agents.middleware import after_model


@after_model
def track_last_response(state, runtime):

    last_message = state["messages"][-1]

    return {
        "last_ai_response": last_message.content
    }
```

Agent state after execution:

```
{
  "messages": [...],
  "model_calls": 3,
  "last_ai_response": "Hello! How can I help you?"
}
```

---


In [9]:
"""
Production-style LangChain Middleware Example
---------------------------------------------

Concepts Covered
- Custom AgentState schema
- Node-style middleware hook (after_model)
- State updates
- Tracking model call count
- Logging responses
- Cheap Claude model
- Real conversation flow

Use Case
--------
Production systems track model usage for:
- cost monitoring
- analytics
- observability
"""

from langchain.agents import create_agent
from langchain.agents.middleware import after_model, AgentState
from langchain_anthropic import ChatAnthropic
from langchain.messages import HumanMessage
from langgraph.runtime import Runtime
from typing import Any
from typing_extensions import NotRequired


# --------------------------------------------------
# Custom Agent State
# --------------------------------------------------


class TrackingState(AgentState):
    """
    Extend the default AgentState to store
    custom tracking metrics.
    """

    model_call_count: NotRequired[int]


# --------------------------------------------------
# Middleware: Track Model Calls
# --------------------------------------------------


@after_model(state_schema=TrackingState)
def increment_after_model(
    state: TrackingState, runtime: Runtime
) -> dict[str, Any] | None:
    """
    Middleware runs after each model response.

    It increments a counter in the agent state
    tracking how many times the model was called.
    """

    current_count = state.get("model_call_count", 0)

    updated_count = current_count + 1

    print(f"\n[Middleware] Model call count → {updated_count}")

    return {"model_call_count": updated_count}


# --------------------------------------------------
# Claude Cheap Model
# --------------------------------------------------

model = ChatAnthropic(model="claude-3-haiku-20240307", temperature=0, max_tokens=20)


# --------------------------------------------------
# Create Agent
# --------------------------------------------------

agent = create_agent(model=model, tools=[], middleware=[increment_after_model])


# --------------------------------------------------
# Conversation Memory
# --------------------------------------------------

chat_history = []


def run_agent(prompt: str):
    """
    Simulate a real conversation with memory.
    """

    print("\nUSER:", prompt)

    chat_history.append(HumanMessage(content=prompt))

    result = agent.invoke({"messages": chat_history})

    response = result["messages"][-1]

    chat_history.append(response)

    print("AI:", response.content)


# --------------------------------------------------
# Test Cases (Real-life style)
# --------------------------------------------------

tests = [
    "Explain machine learning simply",
    "What is LangChain used for?",
    "Explain transformers in AI",
    "What is an AI agent?",
]


for i, prompt in enumerate(tests):

    print("\n-----------------------------")
    print("TEST", i + 1)

    run_agent(prompt)


-----------------------------
TEST 1

USER: Explain machine learning simply

[Middleware] Model call count → 1
AI: Machine learning is a field of artificial intelligence that allows computers to learn and improve from experience without being explicitly

-----------------------------
TEST 2

USER: What is LangChain used for?

[Middleware] Model call count → 1
AI: LangChain is a framework for building applications with large language models (LLMs) like

-----------------------------
TEST 3

USER: Explain transformers in AI

[Middleware] Model call count → 1
AI: Transformers are a type of neural network architecture that has revolutionized the field of natural language

-----------------------------
TEST 4

USER: What is an AI agent?

[Middleware] Model call count → 1
AI: An AI agent is a computer system that is capable of perceiving its environment, making decisions, an



# 2️⃣ Wrap-Style Hooks

Wrap hooks intercept **model calls or tool calls**.

Examples:

- `wrap_model_call`
- `wrap_tool_call`

These hooks allow **full control over execution flow**.

You can:

- retry model calls
- modify inputs
- modify outputs
- track usage
- update state

---

# Wrap Hook Execution Flow

```
User Prompt
     ↓
wrap_model_call
     ↓
Handler (actual model call)
     ↓
Return Response
```

---

# Updating State in Wrap Hooks

Wrap hooks **cannot return a plain dictionary**.

Instead, they must return an **ExtendedModelResponse** that contains a **Command**.

The `Command` object injects updates into the agent state.

---

# Example: Update State From Model Response

Example: **Track token usage**

```python
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langgraph.types import Command
from typing import Callable


@wrap_model_call
def track_usage(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:

    response = handler(request)

    tokens = response.response_metadata.get("token_usage", 0)

    return response.with_command(
        Command(update={"token_usage": tokens})
    )
```

---

# What Happens

1. The model is called normally  
2. Middleware extracts token usage  
3. Middleware injects the state update using `Command`  
4. Agent state now includes token usage  

Example state:

```
{
  "messages": [...],
  "token_usage": 120
}
```

---

# Example: Wrap Tool Call State Update

Wrap hooks can also modify **tool execution state**.

```python
from langchain.agents.middleware import wrap_tool_call
from langgraph.types import Command


@wrap_tool_call
def track_tool_usage(request, handler):

    result = handler(request)

    return Command(
        update={
            "tool_calls": 1
        }
    )
```

This updates the state every time a tool runs.

---

# Real Production Use Cases

State updates are commonly used for:

| Use Case | Example |
|---|---|
| Token tracking | estimate API cost |
| Retry counters | monitor failures |
| Conversation summaries | reduce context size |
| Tool usage analytics | track agent behavior |
| Audit logs | enterprise compliance |

---

# Node Hooks vs Wrap Hooks

| Feature | Node Hooks | Wrap Hooks |
|---|---|---|
| Execution | specific pipeline step | around model/tool call |
| State update | return dict | return Command |
| Control flow | limited | full control |
| Complexity | simple | advanced |

---

# Example Final Agent State

```
{
  "messages": [...],
  "model_calls": 4,
  "token_usage": 380,
  "tool_calls": 2,
  "last_ai_response": "Here is the result."
}
```

Middleware dynamically builds this **state during agent execution**.

---

# Key Takeaway

**Node Hooks**

```python
return {"key": value}
```

**Wrap Hooks**

```python
return response.with_command(
    Command(update={"key": value})
)
```

Node hooks are **simpler**, while wrap hooks provide **full control over execution**.

In [10]:
"""
Production-style LangChain Middleware Example
---------------------------------------------

Concepts Covered
- Custom AgentState schema
- Wrap-style middleware (wrap_model_call)
- ExtendedModelResponse
- Command-based state updates
- Token usage tracking
- Cheap Claude model
- Real conversation flow

Use Case
--------
Production AI systems track token usage to:
- estimate API cost
- monitor model consumption
- build usage analytics
"""

from typing import Callable
from typing_extensions import NotRequired
from typing import Any

from langchain.agents import create_agent
from langchain.agents.middleware import (
    wrap_model_call,
    ModelRequest,
    ModelResponse,
    AgentState,
    ExtendedModelResponse,
)
from langchain.messages import HumanMessage
from langchain_anthropic import ChatAnthropic
from langgraph.types import Command


# --------------------------------------------------
# Custom Agent State
# --------------------------------------------------


class UsageTrackingState(AgentState):
    """
    Extend AgentState to store token usage metrics.
    """

    last_model_call_tokens: NotRequired[int]


# --------------------------------------------------
# Middleware: Track Token Usage
# --------------------------------------------------


@wrap_model_call(state_schema=UsageTrackingState)
def track_usage(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ExtendedModelResponse:
    """
    Wrap-style middleware that intercepts the model call,
    tracks token usage, and updates the agent state.
    """

    print("\n[Middleware] Calling model...")

    response = handler(request)

    # In real production systems, token usage usually comes from:
    # response.response_metadata["token_usage"]
    # Here we simulate it for learning purposes.

    simulated_token_usage = 150

    print(f"[Middleware] Tokens used in this call → {simulated_token_usage}")

    return ExtendedModelResponse(
        model_response=response,
        command=Command(update={"last_model_call_tokens": simulated_token_usage}),
    )


# --------------------------------------------------
# Cheap Claude Model
# --------------------------------------------------

model = ChatAnthropic(
    model="claude-3-haiku-20240307",
    temperature=0,
    max_tokens=20,
)


# --------------------------------------------------
# Create Agent
# --------------------------------------------------

agent = create_agent(
    model=model,
    tools=[],
    middleware=[track_usage],
)


# --------------------------------------------------
# Conversation Memory
# --------------------------------------------------

chat_history = []


def run_agent(prompt: str):
    """
    Simulate a real conversation with memory.
    """

    print("\nUSER:", prompt)

    chat_history.append(HumanMessage(content=prompt))

    result = agent.invoke({"messages": chat_history})

    response = result["messages"][-1]

    chat_history.append(response)

    print("AI:", response.content)


# --------------------------------------------------
# Test Cases
# --------------------------------------------------

tests = [
    "Explain machine learning simply",
    "What is LangChain used for?",
    "Explain transformers in AI",
]


for i, prompt in enumerate(tests):

    print("\n-----------------------------")
    print("TEST", i + 1)

    run_agent(prompt)


-----------------------------
TEST 1

USER: Explain machine learning simply

[Middleware] Calling model...
[Middleware] Tokens used in this call → 150
AI: Machine learning is a field of artificial intelligence that allows computers to learn and improve from experience without being explicitly

-----------------------------
TEST 2

USER: What is LangChain used for?

[Middleware] Calling model...
[Middleware] Tokens used in this call → 150
AI: LangChain is a framework for building applications with large language models (LLMs) like

-----------------------------
TEST 3

USER: Explain transformers in AI

[Middleware] Calling model...
[Middleware] Tokens used in this call → 150
AI: Transformers are a type of neural network architecture that has revolutionized the field of natural language


# LangChain Middleware — Creating Custom Middleware

LangChain allows developers to **extend agent behavior using middleware**.  
Middleware can intercept and modify the agent execution flow at different stages.

You can create middleware in **two main ways**:

1. **Decorator-based middleware**
2. **Class-based middleware**

---

# Middleware Architecture

```
User Input
     ↓
before_agent
     ↓
before_model
     ↓
wrap_model_call
     ↓
Model Call
     ↓
after_model
     ↓
wrap_tool_call
     ↓
Tool Execution
     ↓
after_agent
     ↓
Final Response
```

Middleware hooks allow you to implement:

- logging
- retries
- monitoring
- security checks
- usage tracking
- dynamic prompts

---

# 1️⃣ Decorator-Based Middleware

Decorator middleware is **quick and simple**.  
It is ideal for **single-hook middleware**.

You simply wrap a function with a middleware decorator.

---

## Available Decorators

### Node-Style Hooks

These run at specific points in the execution pipeline.

| Decorator | Description |
|---|---|
| `@before_agent` | Runs once before the agent starts |
| `@before_model` | Runs before every model call |
| `@after_model` | Runs after every model response |
| `@after_agent` | Runs once after the agent finishes |

---

### Wrap-Style Hooks

These **wrap around model or tool execution** and give full control.

| Decorator | Description |
|---|---|
| `@wrap_model_call` | Intercepts and controls model calls |
| `@wrap_tool_call` | Intercepts and controls tool calls |

---

### Convenience Decorator

| Decorator | Description |
|---|---|
| `@dynamic_prompt` | Dynamically generate system prompts |

---

# Example — Decorator Middleware

```python
from langchain.agents.middleware import after_model
from langgraph.runtime import Runtime


@after_model
def log_response(state, runtime):

    print("Model response:")
    print(state["messages"][-1].content)

    return None
```

This middleware **logs every model response**.

---

# When to Use Decorator Middleware

Use decorators when:

- Only **one hook** is needed
- No **complex configuration** is required
- You want **quick prototyping**
- Middleware logic is **simple**

Examples:

- logging responses
- basic validation
- debugging

---

# 2️⃣ Class-Based Middleware

Class-based middleware is **more powerful and flexible**.

It allows you to:

- combine multiple hooks
- define configuration in `__init__`
- reuse middleware across projects
- support both sync and async implementations

---

# Example — Class-Based Middleware

```python
from langchain.agents.middleware import AgentMiddleware
from langchain.agents.middleware import ModelRequest, ModelResponse
from typing import Callable


class RetryMiddleware(AgentMiddleware):

    def __init__(self, max_retries: int = 3):
        super().__init__()
        self.max_retries = max_retries


    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse:

        for attempt in range(self.max_retries):

            try:
                return handler(request)

            except Exception as e:

                if attempt == self.max_retries - 1:
                    raise

                print(f"Retry {attempt + 1}/{self.max_retries} after error: {e}")
```

This middleware **automatically retries failed model calls**.

---

# When to Use Class Middleware

Use class-based middleware when:

- Multiple hooks are required
- Complex configuration is needed
- You want reusable middleware
- Both **sync and async implementations** are required
- Middleware needs **stateful behavior**

Examples:

- retry middleware
- cost tracking middleware
- security filters
- rate limiting

---

# Custom State Schema

Middleware can extend the **agent state** with custom properties.

This allows middleware to maintain information throughout the agent execution lifecycle.

---

# Why Custom State Is Useful

Custom state enables middleware to:

### 1️⃣ Track Execution State

Maintain counters or flags across execution.

Example:

- number of model calls
- number of tool executions

---

### 2️⃣ Share Data Between Hooks

State can pass data between hooks.

Example:

```
before_model → store timestamp
after_model → calculate latency
```

---

### 3️⃣ Implement Cross-Cutting Concerns

Middleware can add functionality without modifying agent logic.

Examples:

- rate limiting
- token usage tracking
- audit logging
- user session tracking

---

### 4️⃣ Make Conditional Decisions

Middleware can modify execution behavior based on accumulated state.

Examples:

- stop agent after token limit
- skip tool calls
- route to different models

---

# Example — Custom State Schema

```python
from langchain.agents.middleware import AgentState
from typing_extensions import NotRequired


class UsageState(AgentState):

    token_usage: NotRequired[int]
```

Now middleware can store token usage inside the state.

---

# Production Middleware Example

Common middleware stack in production systems:

```
Security Middleware
      ↓
Rate Limit Middleware
      ↓
Retry Middleware
      ↓
Cost Tracking Middleware
      ↓
Model Call
      ↓
Logging Middleware
```

This architecture helps build **robust AI agents**.

---

# Summary

| Feature | Decorator Middleware | Class Middleware |
|---|---|---|
| Complexity | Simple | Advanced |
| Hooks | Usually one | Multiple |
| Configuration | Limited | Flexible |
| Reusability | Low | High |
| Use Case | Prototyping | Production systems |

---

# Key Takeaway

- **Decorator middleware** → simple and quick
- **Class middleware** → powerful and reusable
- **Custom state schema** → enables advanced tracking and control

These concepts are essential when building **production-ready LangChain agents**.